# AGH-HardNet Colab Notebook

Bu notebook, AGH-Former ciktilari uzerinden yalnizca `LM0`, `LM21` ve `LM22` icin HardNet specialist refiner calistirir.

Sirayla calistirma onerisi:

1. Drive mount ve repo guncelleme
2. Bagimlilik kurulumu
3. Yol kontrolu
4. Oracle analizi
5. Full hard3 kosusu
6. Trichion ve Gonion specialist kosulari
7. Specialist sonuclari birlestirme


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'

if not CODE_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)

print('CODE_ROOT:', CODE_ROOT)

In [ ]:
%cd /content/comparative-study/agh_hardnet_refiner
!python -m pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/orthodontic')
DATA_ROOT = DRIVE_ROOT / 'data' / 'dataset'
RUN_ROOT = DRIVE_ROOT / 'hardnet_runs'

# Gerekirse bu yolu kendi Drive klasor adina gore degistir.
AGH_RUN = DRIVE_ROOT / 'diffusion_runs' / 'aghformer_v6_stage2_raw_fine_refiner_p12000'

print('DATA_ROOT exists:', DATA_ROOT.exists(), DATA_ROOT)
print('AGH_RUN exists:', AGH_RUN.exists(), AGH_RUN)
print('val predictions:', (AGH_RUN / 'refined_predictions_val.csv').exists())
print('test predictions:', (AGH_RUN / 'refined_predictions_test.csv').exists())
print('train refined predictions:', (AGH_RUN / 'refined_predictions_train.csv').exists())
print('train stage1 predictions:', (AGH_RUN / 'stage1_predictions_train.csv').exists())

## 1. Oracle Analizi

Bu hucre, LM0/21/22 icin patch icinde uzmana yakin aday nokta olup olmadigini olcer. Oracle iyi ise problem aday secimi problemidir.

In [ ]:
!python -u colab_run_agh_hardnet.py \
  --preset oracle \
  --agh-run "$AGH_RUN" \
  --data-root "$DATA_ROOT" \
  --run-root "$RUN_ROOT"

## 2. Full Hard3 Kosusu

LM0, LM21 ve LM22 icin tek ortak specialist model egitir.

In [ ]:
!python -u colab_run_agh_hardnet.py \
  --preset full \
  --agh-run "$AGH_RUN" \
  --data-root "$DATA_ROOT" \
  --run-root "$RUN_ROOT"

## 3. Ayrı Specialist Kosular

`trichion` yalniz LM0'i, `gonion` yalniz LM21/22'yi refine eder.

In [ ]:
!python -u colab_run_agh_hardnet.py \
  --preset trichion \
  --agh-run "$AGH_RUN" \
  --data-root "$DATA_ROOT" \
  --run-root "$RUN_ROOT"

In [ ]:
!python -u colab_run_agh_hardnet.py \
  --preset gonion \
  --agh-run "$AGH_RUN" \
  --data-root "$DATA_ROOT" \
  --run-root "$RUN_ROOT"

## 4. Specialist Sonuclari Birlestirme

In [ ]:
!python -u merge_specialist_predictions.py \
  --base-dir "$RUN_ROOT/agh_hardnet_trichion_candidate" \
  --specialist-dirs "$RUN_ROOT/agh_hardnet_gonion_candidate" \
  --output-dir "$RUN_ROOT/agh_hardnet_merged_specialists"

## 5. Metrikleri Oku

In [ ]:
import json
from pathlib import Path

for path in [
    RUN_ROOT / 'agh_hardnet_full' / 'metrics_hardnet.json',
    RUN_ROOT / 'agh_hardnet_trichion_candidate' / 'metrics_hardnet.json',
    RUN_ROOT / 'agh_hardnet_gonion_candidate' / 'metrics_hardnet.json',
    RUN_ROOT / 'agh_hardnet_merged_specialists' / 'metrics_merged_specialists.json',
]:
    print('\n', path)
    if not path.exists():
        print('not found')
        continue
    data = json.loads(path.read_text())
    if 'hardnet' in data:
        print('base test ALE:', data['base']['test']['overall']['ale'])
        print('hardnet test ALE:', data['hardnet']['test']['overall']['ale'])
        print('base hard3 ALE:', data['base']['test']['hard_landmarks']['ale'])
        print('hardnet hard3 ALE:', data['hardnet']['test']['hard_landmarks']['ale'])
    else:
        print('merged test ALE:', data['metrics']['test']['overall']['ale'])
        print('merged hard3 ALE:', data['metrics']['test']['hard_landmarks']['ale'])